In [ ]:
# Исследование датасета изображений модерации (EDA + визуализация + проверки)

from __future__ import annotations

from io import BytesIO
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

sns.set_theme(style='whitegrid')

import sys
sys.path.append(str(Path('..').resolve() / 'src'))

from pinz_ml.paths import data_dir

DATA_PATH = data_dir() / 'images_dataset.parquet'
assert DATA_PATH.exists(), f"Не найден parquet: {DATA_PATH}"

df = pd.read_parquet(DATA_PATH)
print('Rows:', len(df))
print('Columns:', list(df.columns))
display(df.head(10))

has_bytes = 'image_bytes' in df.columns
has_path = 'image_path' in df.columns
print('has_bytes:', has_bytes, 'has_path:', has_path)

assert has_bytes or has_path, "Нужна колонка image_bytes или image_path"

# Пропуски
missing = df.isna().mean().sort_values(ascending=False)
print('Missing ratio (top 20):')
display(missing.head(20))

# Распределение label
if 'label' in df.columns:
    plt.figure(figsize=(6, 3))
    df['label'].value_counts().sort_index().plot(kind='bar')
    plt.title('Label distribution (images)')
    plt.tight_layout()
    plt.show()


def load_pil(row) -> Image.Image:
    if has_bytes and row.get('image_bytes') is not None:
        return Image.open(BytesIO(row['image_bytes'])).convert('RGB')
    if has_path and row.get('image_path'):
        return Image.open(row['image_path']).convert('RGB')
    raise ValueError('bad row (no bytes/path)')

# Визуализация случайных картинок
n = min(12, len(df))
rows = df.sample(n, random_state=42)
fig, axes = plt.subplots(3, 4, figsize=(12, 8))
axes = axes.flatten()
for ax, (_, r) in zip(axes, rows.iterrows()):
    try:
        im = load_pil(r)
        ax.imshow(im)
        ax.set_title(f"label={int(r.get('label', -1))}")
    except Exception as e:
        ax.text(0.05, 0.5, str(e))
    ax.axis('off')
plt.tight_layout()
plt.show()

# Размеры/аспект
sizes = []
for _, r in df.head(400).iterrows():
    try:
        im = load_pil(r)
        w, h = im.size
        sizes.append((w, h))
    except Exception:
        continue

if sizes:
    wh = np.array(sizes)
    df_sizes = pd.DataFrame({'w': wh[:, 0], 'h': wh[:, 1], 'aspect': wh[:, 0] / np.maximum(1, wh[:, 1])})
    fig, ax = plt.subplots(1, 3, figsize=(12, 3))
    sns.histplot(df_sizes['w'], bins=40, ax=ax[0]); ax[0].set_title('width')
    sns.histplot(df_sizes['h'], bins=40, ax=ax[1]); ax[1].set_title('height')
    sns.histplot(df_sizes['aspect'], bins=40, ax=ax[2]); ax[2].set_title('aspect')
    plt.tight_layout(); plt.show()

    print('Size quantiles:')
    display(df_sizes[['w', 'h', 'aspect']].quantile([0.5, 0.9, 0.95, 0.99]))


In [ ]:
# Простые преобразования изображений (пример аугментаций)

from torchvision import transforms

# Берём 1 картинку и смотрим, как меняется после аугментаций
row = df.sample(1, random_state=1).iloc[0]
im = load_pil(row)

aug = transforms.Compose(
    [
        transforms.Resize((256, 256)),
        transforms.RandomResizedCrop((224, 224), scale=(0.7, 1.0)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    ]
)

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
axes = axes.flatten()
axes[0].imshow(im); axes[0].set_title('original'); axes[0].axis('off')

for i in range(1, 8):
    out = aug(im)
    axes[i].imshow(out)
    axes[i].set_title(f'aug {i}')
    axes[i].axis('off')

plt.tight_layout()
plt.show()
